In [6]:
import requests
from pathlib import Path
subhalo_id = 340851
PROJECT_DIR = Path(
    r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1"
)

output_dir = PROJECT_DIR / "data" / "TNG300-1" / "output"
output_dir.mkdir(parents=True, exist_ok=True)

tree_file = output_dir / f"tree_{subhalo_id}.hdf5"


print("Downloaded merger tree to:")
print(tree_file.resolve())
API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c "



url = (
    f"https://www.tng-project.org/api/"
    f"TNG300-1/snapshots/99/subhalos/{subhalo_id}/sublink/mpb.hdf5"
)

response = requests.get(
    url,
    headers={"api-key": API_KEY}
)

response.raise_for_status()

with open(tree_file, "wb") as f:
    f.write(response.content)

print("Downloaded merger tree.")

Downloaded merger tree to:
C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\TNG300-1\output\tree_340851.hdf5
Downloaded merger tree.


In [ ]:
import h5py

tree_file = r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1\data\TNG300-1\output\tree_738558.hdf5"

with h5py.File(tree_file, "r") as f:
    print("Top-level contents:")
    print(list(f.keys()))

    for key in f.keys():
        obj = f[key]
        print(f"\n{key}:")
        print("  shape:", getattr(obj, "shape", None))
        print("  dtype:", getattr(obj, "dtype", None))

Top-level contents:
['DescendantID', 'FirstProgenitorID', 'FirstSubhaloInFOFGroupID', 'GroupBHMass', 'GroupBHMdot', 'GroupCM', 'GroupFirstSub', 'GroupGasMetalFractions', 'GroupGasMetallicity', 'GroupLen', 'GroupLenType', 'GroupMass', 'GroupMassType', 'GroupNsubs', 'GroupPos', 'GroupSFR', 'GroupStarMetalFractions', 'GroupStarMetallicity', 'GroupVel', 'GroupWindMass', 'Group_M_Crit200', 'Group_M_Crit500', 'Group_M_Mean200', 'Group_M_TopHat200', 'Group_R_Crit200', 'Group_R_Crit500', 'Group_R_Mean200', 'Group_R_TopHat200', 'LastProgenitorID', 'MainLeafProgenitorID', 'Mass', 'MassHistory', 'NextProgenitorID', 'NextSubhaloInFOFGroupID', 'NumParticles', 'RootDescendantID', 'SnapNum', 'SubfindID', 'SubhaloBHMass', 'SubhaloBHMdot', 'SubhaloBfldDisk', 'SubhaloBfldHalo', 'SubhaloCM', 'SubhaloGasMetalFractions', 'SubhaloGasMetalFractionsHalfRad', 'SubhaloGasMetalFractionsMaxRad', 'SubhaloGasMetalFractionsSfr', 'SubhaloGasMetalFractionsSfrWeighted', 'SubhaloGasMetallicity', 'SubhaloGasMetallicityHa

In [ ]:

import matplotlib.pyplot as plt
fig, axes = plt.subplots(
    1,
    1,
    figsize=(8, 5),
)

for ax, subhalo_id in zip(axes, plot_ids):


    profile_file = (
        profile_dir /
        f"subhalo_{subhalo_id}_mass_profile.csv"
    )

    profile = pd.read_csv(profile_file)

    dm_mask = profile["r_DM_kpc"].notna()

    r_dm = profile.loc[
        dm_mask,
        "r_DM_kpc"
    ].to_numpy()

    m_dm = profile.loc[
        dm_mask,
        "M_DM_cumulative_1e10Msun_h"
    ].to_numpy()
    m_dm = m_dm * 1e10 / h

    star_mask = profile["r_star_kpc"].notna()

    r_star = profile.loc[
        star_mask,
        "r_star_kpc"
    ].to_numpy()

    m_star = profile.loc[
        star_mask,
        "M_star_cumulative_1e10Msun_h"
    ].to_numpy()

    m_star = m_star * 1e10 / h


    all_radii = np.unique(
        np.concatenate([
            r_dm,
            r_star,
        ])
    )
    total_dm = np.interp(
        all_radii,
        r_dm,
        m_dm,
        left=0,
        right=m_dm[-1]
    )

    total_star = np.interp(
        all_radii,
        r_star,
        m_star,
        left=0,
        right=m_star[-1]
    )



    total_gas = np.zeros_like(all_radii)

    total_mass = (
        total_dm
        + total_star
        + total_gas
    )

    row = catalog[
        catalog["SubhaloID"] == subhalo_id
    ].iloc[0]

    Rh = row["R_half_star"] / h

    ax.plot(
        r_dm,
        m_dm,
        color="black",
        linewidth=2,
        label="DM"
    )

    ax.plot(
        r_star,
        m_star,
        color="red",
        linewidth=2,
        label="Star"
    )

    ax.plot(
        all_radii,
        total_mass,
        color="gray",
        linewidth=2,
        label="Total"
    )

    ax.axvline(
        Rh,
        color="green",
        linestyle=":",
        linewidth=2
    )

    ax.set_xscale("linear")

    ax.set_yscale("log")

    ax.set_xlim(0, 5)

    ax.set_ylim(
        1e6,
        1e10
    )

    ax.set_xlabel(r"$r$ [kpc]")

    ax.set_title(
        f"Subhalo {subhalo_id}"
    )

    ax.grid(
        True,
        which="both",
        alpha=0.2
    )
axes[0].set_ylabel(
    r"$M(<r)$ [$M_\odot$]"
)
axes[0].legend(
    loc="upper left",
    frameon=False
)

plt.tight_layout()

output_file = (
    project_dir
    / "figures"
    / "mass_profiles_two_panel.png"
)

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
for ax in axes[len(plot_ids):]:
    ax.set_visible(False)
print()
print("Saved:")
print(output_file)